# Single-tool eval: `list_services`

Runs the real production agent (`agent.agent.get_agent()`) restricted to **only** the `list_services` tool from `agent.tools.docker`. Everything else the full agent could reach for is unavailable for this call — same system prompt, middleware, model, and checkpointer the Discord bot runs, just a narrower tool set. This measures how far the deployed agent gets on this backend alone.


In [5]:
from eval_transcript import run_agent_eval
from agent.tools.docker import list_services

In [6]:
TOOLS = [list_services]

### The loop

Every run below goes through `eval_transcript.run_agent_eval`, which drives the real production agent (`agent.agent.get_agent()`) — the same system prompt, middleware, model, and checkpointer the Discord bot runs — restricted to `TOOLS` for this call. It prints each tool call and dumps the full transcript to `eval/logs/<timestamp>__<label>.json`.


## In-scope question

A question `list_services` alone should be able to answer.


In [7]:
answer = await run_agent_eval(
    "01_list_services:live-inscope",
    "What containers and monitored endpoints exist in this lab?",
    thread_id="01_list_services:live-inscope",
    tools=TOOLS,
)
print(answer)


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


[transcript] 01_list_services:live-inscope: 2 new message(s), 0 tool call(s), 8145ms — tokens in=1465 out=219 total=1684
[transcript] saved to eval/logs/20260827T102841892511__01_list_services-live-inscope.json
**Containers**
* `marconi-agent-api`
* `marconi-agent-bot`
* `alertmanager`
* `grafana`
* `prometheus`
* `promtail`
* `loki`
* `mlbox-gpu0`
* `cadvisor`
* `node-exporter`

**Monitored Endpoints**
* `prometheus`: `http://localhost:9090/-/healthy`
* `loki`: `http://localhost:3100/ready`
* `grafana`: `http://localhost:3000/api/health`
* `alertmanager`: `http://localhost:9093/-/healthy`


## Out-of-scope question

A question that needs evidence outside this backend. Watch whether the model admits the limit or hallucinates past it.


In [8]:
answer = await run_agent_eval(
    "01_list_services:live-outofscope",
    "Is the agent-api container currently healthy?",
    thread_id="01_list_services:live-outofscope",
    tools=TOOLS,
)
print(answer)


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


[transcript] 01_list_services:live-outofscope: 2 new message(s), 0 tool call(s), 9151ms — tokens in=4396 out=256 total=4652
[transcript] saved to eval/logs/20260827T102851110852__01_list_services-live-outofscope.json
I have identified the service as `marconi-agent-api`, but I lack the tools to verify its health or current status.

**Likely Cause**: Toolset limitation.
**Recommended Action**: Manually check the status of `marconi-agent-api` using `docker ps`.
